# Complete Batch

![Incremental Data Processing](../../z-course-images/formula1-incremental-data-processing.png "Incremental Data Processing")

In [0]:
%run ../00-common/01.environment_config

In [0]:
control_table = f"{catalog_name}.{control_schema}.batch_control"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql import Row

In [0]:
if v_batch_id:
    delta_table = DeltaTable.forName(spark,control_table)

    source_df = (
        spark.createDataFrame([Row(batch_id=v_batch_id)])
        .withColumn('status',lit('completed'))
        .withColumn('updated_timestamp',current_timestamp())
    )

    (
        delta_table.alias('t')
        .merge(source_df.alias('s'),"t.batch_id=s.batch_id AND t.status='in_progress'" )
        .whenMatchedUpdate(set={"t.status":"s.status","t.updated_timestamp":"s.updated_timestamp"})
        .execute()
    )
    print(f"Marked batch {v_batch_id} as completed")
else:
    raise Exception("batch_id is missing")  

In [0]:
display(spark.table(control_table))